# SO101 ACT Evaluation in MuJoCo
Load trained ACT model, run inference in MuJoCo SO101 simulation, render video.
Requires Internet + GPU T4.

In [ ]:
!curl -s https://ipinfo.io 2>/dev/null && echo 'INTERNET OK' || echo 'INTERNET FAILED'

In [ ]:
import os
os.environ.pop('HF_ENDPOINT', None)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['MUJOCO_GL'] = 'egl'

HF_TOKEN = 'hf_TOKEN_REMOVED'
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_REPO = 'xieyucheng123/so101-act'
DATASET_REPO = 'lerobot/svla_so101_pickplace'
NUM_EPISODES = 3
MAX_STEPS = 200
print(f'Config ready, token len={len(HF_TOKEN)}')

In [ ]:
!pip install -q 'lerobot[dataset]' mujoco opencv-python-headless 2>&1 | tail -15

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import lerobot
print(f'lerobot version: {getattr(lerobot, "__version__", "unknown")}')
import mujoco
print(f'mujoco version: {mujoco.__version__}')

## Clone SO-ARM100 and setup MuJoCo scene with cameras

In [ ]:
import subprocess
result = subprocess.run(['git', 'clone', '--depth=1',
    'https://github.com/TheRobotStudio/SO-ARM100.git',
    '/kaggle/working/SO-ARM100'], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'clone done')
if result.returncode != 0:
    print(f'STDERR: {result.stderr[-500:]}')

In [ ]:
SO101_DIR = '/kaggle/working/SO-ARM100/Simulation/SO101'
import os
for f in os.listdir(SO101_DIR):
    print(f)

In [ ]:
# Create scene with cameras for rendering
scene_xml = '''<mujoco model="scene_eval">
    <include file="so101_new_calib.xml" />

    <visual>
        <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0" />
        <rgba haze="0.15 0.25 0.35 1" />
        <global azimuth="160" elevation="-20" />
    </visual>

    <asset>
        <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512"
            height="3072" />
        <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4"
            rgb2="0.1 0.2 0.3"
            markrgb="0.8 0.8 0.8" width="300" height="300" />
        <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5"
            reflectance="0.2" />
    </asset>

    <worldbody>
        <light pos="0 0 3.5" dir="0 0 -1" directional="true" />
        <geom name="floor" size="0 0 0.05" pos="0 0 0" type="plane" material="groundplane" />
        <!-- overhead camera (up) -->
        <camera name="up" pos="0.0 -0.15 0.55" quat="0.707 0 0 0.707" fovy="45" />
        <!-- side camera -->
        <camera name="side" pos="0.45 0.0 0.25" quat="0.5 0 0.5 -0.5" fovy="45" />
    </worldbody>
</mujoco>
'''

scene_path = os.path.join(SO101_DIR, 'scene_eval.xml')
with open(scene_path, 'w') as f:
    f.write(scene_xml)
print(f'Scene written to {scene_path}')

In [ ]:
# Test MuJoCo loading
model_mj = mujoco.MjModel.from_xml_path(scene_path)
data_mj = mujoco.MjData(model_mj)
print(f'MuJoCo model loaded: nq={model_mj.nq}, nu={model_mj.nu}')
print(f'Joint names: {[model_mj.joint(i).name for i in range(model_mj.njnt)]}')
print(f'Actuator names: {[model_mj.actuator(i).name for i in range(model_mj.nu)]}')

# Test rendering
renderer = mujoco.Renderer(model_mj, height=480, width=640)
mujoco.mj_forward(model_mj, data_mj)
renderer.update_scene(data_mj, camera='up')
img = renderer.render()
print(f'Image shape: {img.shape}, dtype: {img.dtype}')

## Load ACT policy and dataset metadata

In [ ]:
from lerobot.policies.act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.utils import build_inference_frame
from lerobot.datasets import LeRobotDatasetMetadata
import torch

device = torch.device('cuda')

print(f'Loading model from {MODEL_REPO}...')
policy = ACTPolicy.from_pretrained(MODEL_REPO)
policy.to(device)
policy.eval()
print('Model loaded!')

In [ ]:
print(f'Loading dataset metadata from {DATASET_REPO}...')
dataset_metadata = LeRobotDatasetMetadata(DATASET_REPO)
print(f'Dataset features: {list(dataset_metadata.features.keys())}')
print(f'Dataset stats keys: {list(dataset_metadata.stats.keys())}')

In [ ]:
# Create pre/post processors
try:
    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        pretrained_path=MODEL_REPO,
        dataset_stats=dataset_metadata.stats,
    )
    print('Processors created with pretrained_path')
except Exception as e:
    print(f'Failed with pretrained_path: {e}')
    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        dataset_stats=dataset_metadata.stats,
    )
    print('Processors created with dataset_stats only')

In [ ]:
# Print model config for debugging
print(f'Policy config type: {type(policy.config).__name__}')
print(f'chunk_size: {policy.config.chunk_size}')
print(f'n_action_steps: {policy.config.n_action_steps}')
if hasattr(policy.config, 'input_features'):
    print(f'Input features: {policy.config.input_features}')
if hasattr(policy.config, 'output_features'):
    print(f'Output features: {policy.config.output_features}')

## Run inference in MuJoCo simulation

In [ ]:
import numpy as np
import cv2

model_mj = mujoco.MjModel.from_xml_path(scene_path)
data_mj = mujoco.MjData(model_mj)
renderer = mujoco.Renderer(model_mj, height=480, width=640)

all_frames = []
episode_results = []

for ep in range(NUM_EPISODES):
    print(f'\n=== Episode {ep+1}/{NUM_EPISODES} ===')
    mujoco.mj_resetData(model_mj, data_mj)
    
    # Reset policy queues
    if hasattr(policy, 'reset'):
        policy.reset()
    
    ep_frames = []
    actions_taken = []
    
    for step in range(MAX_STEPS):
        # Get joint positions from MuJoCo (6 DOF)
        state = data_mj.qpos[:6].copy()
        
        # Render camera images
        renderer.update_scene(data_mj, camera='up')
        img_up = renderer.render()  # (H, W, 3) uint8
        renderer.update_scene(data_mj, camera='side')
        img_side = renderer.render()
        
        # Build observation dict matching robot format (joint names + camera names)
        joint_names = ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
        obs = {}
        for i, name in enumerate(joint_names):
            obs[f'{name}.pos'] = state[i]
        obs['up'] = img_up
        obs['side'] = img_side
        
        # Build inference frame and preprocess
        try:
            obs_frame = build_inference_frame(
                observation=obs,
                ds_features=dataset_metadata.features,
                device=device,
            )
            obs_processed = preprocessor(obs_frame)
        except Exception as e:
            if step == 0:
                print(f'  Preprocess error at step 0: {e}')
            break
        
        # Policy inference
        with torch.no_grad():
            action = policy.select_action(obs_processed)
            action = postprocessor(action)
        
        # Convert to numpy
        if isinstance(action, torch.Tensor):
            action_np = action.cpu().numpy()
        else:
            action_np = np.array(action)
        
        action_flat = action_np.flatten()[:6]
        actions_taken.append(action_flat.copy())
        
        # Apply action to MuJoCo (position control)
        data_mj.ctrl[:6] = action_flat
        mujoco.mj_step(model_mj, data_mj)
        
        # Render frame for video (use 'up' camera)
        renderer.update_scene(data_mj, camera='up')
        frame = renderer.render()
        ep_frames.append(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        
        if step % 50 == 0:
            print(f'  Step {step}: state={state[:3].round(3)}, action={action_flat[:3].round(3)}')
    
    all_frames.extend(ep_frames)
    actions_arr = np.array(actions_taken)
    episode_results.append({
        'episode': ep + 1,
        'steps': len(ep_frames),
        'action_mean': actions_arr.mean(axis=0).tolist() if len(actions_arr) > 0 else None,
        'action_std': actions_arr.std(axis=0).tolist() if len(actions_arr) > 0 else None,
    })
    print(f'  Completed {len(ep_frames)} steps')

print(f'\nTotal frames: {len(all_frames)}')

## Save evaluation video

In [ ]:
video_path = '/kaggle/working/eval_video.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(video_path, fourcc, 30, (640, 480))
for frame in all_frames:
    writer.write(frame)
writer.release()

import os
size_mb = os.path.getsize(video_path) / 1024 / 1024
print(f'Video saved: {video_path} ({size_mb:.1f} MB, {len(all_frames)} frames)')

In [ ]:
# Save evaluation report
import json
report = {
    'model_repo': MODEL_REPO,
    'dataset_repo': DATASET_REPO,
    'num_episodes': NUM_EPISODES,
    'max_steps': MAX_STEPS,
    'episodes': episode_results,
    'video_path': video_path,
    'total_frames': len(all_frames),
}
report_path = '/kaggle/working/eval_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Report saved: {report_path}')
print(json.dumps(report, indent=2))

## Upload video to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
VIDEO_REPO = 'xieyucheng123/so101-act'

api.upload_file(
    path_or_fileobj=video_path,
    path_in_repo='eval_videos/eval_video.mp4',
    repo_id=VIDEO_REPO,
    repo_type='model',
    token=HF_TOKEN,
)
print(f'Video uploaded to {VIDEO_REPO}/eval_videos/eval_video.mp4')

In [ ]:
api.upload_file(
    path_or_fileobj=report_path,
    path_in_repo='eval_videos/eval_report.json',
    repo_id=VIDEO_REPO,
    repo_type='model',
    token=HF_TOKEN,
)
print(f'Report uploaded to {VIDEO_REPO}/eval_videos/eval_report.json')

In [ ]:
print('=== Evaluation Complete ===')
print(f'Model: {MODEL_REPO}')
print(f'Episodes: {NUM_EPISODES}, Steps per episode: {MAX_STEPS}')
print(f'Total frames: {len(all_frames)}')
print(f'Video: {video_path}')
print(f'Uploaded to: {VIDEO_REPO}/eval_videos/')